# Dataset Adequacy Analysis — GMU Cherry Japan (Yedoensis)

Investigates whether the `GMU_Cherry_Japan_Y` dataset provides sufficient
coverage to fit a **chill-forcing (CF)** phenology model for *Prunus yedoensis*.

Key questions:
1. **Spatial coverage** — do the 67 stations span the full latitudinal and
   climatic range of Japan?
2. **Regional balance** — are all regions (Hokkaido → Okinawa) represented?
3. **Temporal completeness** — are all stations active in all years?
4. **Driver range** — do chilling and forcing accumulations vary enough to
   constrain the model's threshold parameters?
5. **Temporal variation** — do the observed years span meaningfully different
   climates?

Set `USE_SYNTHETIC = True` to replace the real dataset with controlled synthetic
data for debugging or to demonstrate adequate/inadequate coverage.

## Config

In [ ]:
# ── toggle real vs synthetic ───────────────────────────────────────────────
USE_SYNTHETIC        = False
DOWNLOAD_TEMPERATURE = True    # download OpenMeteo temperature; False to skip

# Synthetic-mode settings
SYN_N_LOCS      = 50
SYN_N_YEARS     = 30
SYN_CONFOUNDED  = True    # True  → chilling varies only with latitude
                           # False → independent chilling and forcing variation

# ── dataset settings ──────────────────────────────────────────────────────
DATASET_KEY = 'GMU_Cherry_Japan_Y'
OBS_KEY     = 'gmu_0'

# ── binning ───────────────────────────────────────────────────────────────
LAT_BINS      = [24, 30, 35, 40, 45, 46]
LAT_LABELS    = ['24–30°N', '30–35°N', '35–40°N', '40–45°N', '45–46°N']
CLIMATE_BINS  = [-5, 0, 2, 4, 6, 8, 10, 15]
CLIMATE_LABELS = ['<0°C', '0–2°C', '2–4°C', '4–6°C', '6–8°C', '8–10°C', '>10°C']
SPRING_BINS   = [4, 8, 10, 12, 14, 16, 18, 24]
SPRING_LABELS = ['<8°C', '8–10°C', '10–12°C', '12–14°C', '14–16°C', '16–18°C', '>18°C']


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from pysephone.constants import KEY_FEATURES
from pysephone.dataset.dataset import Dataset
from pysephone.dataset.util.calendar import Calendar
from pysephone.dataset.util.openmeteo import OpenMeteoFeatures
from pysephone.models.util.func_phenology import func_chilling_days
from pysephone.data.gmu_cherry.regions_data import (
    LOCATIONS_REGIONS_JAPAN, REGIONS_JAPAN,
)

def _region(loc_id: str) -> str:
    key = loc_id.replace('__', '/')
    rid = LOCATIONS_REGIONS_JAPAN.get(key)
    return REGIONS_JAPAN.get(rid, 'Unknown') if rid else 'Unknown'

REGION_ORDER = [
    'Hokkaido', 'Tohoku', 'Kanto-Koshin', 'Hokuriku', 'Tokai',
    'Kinki', 'Chugoku', 'Shikoku', 'Kyushu-North',
    'Kyushu-South-Amami', 'Okinawa', 'Unknown',
]
REGION_PALETTE = dict(zip(
    REGION_ORDER, sns.color_palette('tab20', len(REGION_ORDER))
))


## 1. Load data

In [ ]:
if not USE_SYNTHETIC:
    cal      = Calendar(default_start='10-01', default_length=365)
    features = OpenMeteoFeatures(calendar=cal) if DOWNLOAD_TEMPERATURE else None
    ds  = Dataset.load(DATASET_KEY, calendar=cal,
                       feature_providers=[features] if features else [])
    obs = ds.observations

    rows = []
    for item in ds.iter_items():
        if OBS_KEY not in item.get('observations_index', {}):
            continue
        rows.append({
            'src':      item['src'],
            'loc_id':   item['loc_id'],
            'year':     item['year'],
            'bloom_ix': int(item['observations_index'][OBS_KEY]),
            'lat':      item['lat'],
            'lon':      item['lon'],
        })
    df = pd.DataFrame(rows)

    locs = obs._df_y_loc.reset_index()
    df = df.merge(locs[['src','loc_id','alt','loc_name']], on=['src','loc_id'], how='left')
    df['region']   = df['loc_id'].apply(_region)
    df['loc_short'] = df['loc_name'].str.replace('Japan/', '', regex=False)

    print(f'Loaded {DATASET_KEY}')
    print(f'  {len(df)} observations  |  {df["loc_id"].nunique()} locations  |  '
          f'{df["year"].nunique()} years')
    print(f'  Bloom_ix (days from Oct 1): {df["bloom_ix"].min()} – {df["bloom_ix"].max()}')

else:
    rng = np.random.default_rng(0)
    lats = rng.uniform(26, 45, SYN_N_LOCS)
    lons = rng.uniform(124, 145, SYN_N_LOCS)
    alts = rng.uniform(0, 300, SYN_N_LOCS)
    regions_syn = ['Hokkaido','Tohoku','Kanto-Koshin','Kinki','Okinawa']
    rows = []
    for i, (lat, lon, alt) in enumerate(zip(lats, lons, alts)):
        mean_wT = 12 - (lat - 35) * 0.7 + rng.normal(0, 0.5)
        mean_sT = 16 - (lat - 35) * 0.4 + rng.normal(0, 0.5)
        chill   = 50 + (lat - 35) * 6 + rng.normal(0, 5)
        fgdu    = 900 - (lat - 35) * 25 + rng.normal(0, 40)
        base_bx = 130 + (lat - 35) * 5
        region  = regions_syn[min(int((lat - 26) / 4), 4)]
        for year in range(2000, 2000 + SYN_N_YEARS):
            yr_a = rng.normal(0, 3)
            bx   = int(base_bx + yr_a * 2 + rng.normal(0, 2))
            rows.append({
                'src': 'synthetic', 'loc_id': f'loc_{i}', 'year': year,
                'bloom_ix': bx, 'lat': lat, 'lon': lon, 'alt': alt,
                'region': region, 'loc_short': f'Loc{i}',
                'mean_winter_T': mean_wT + yr_a * 0.3,
                'mean_spring_T': mean_sT - yr_a * 0.2,
                'chill_days': max(0, chill - yr_a * 2),
                'forcing_gdu': max(0, fgdu + yr_a * 10),
            })
    df = pd.DataFrame(rows)
    print(f'Generated {len(df)} synthetic samples '
          f'({"confounded" if SYN_CONFOUNDED else "balanced"})')

df['lat_bin'] = pd.cut(df['lat'], bins=LAT_BINS, labels=LAT_LABELS)
HAS_TEMP = USE_SYNTHETIC


In [ ]:
# ── Temperature download ─────────────────────────────────────────────────
if DOWNLOAD_TEMPERATURE and not USE_SYNTHETIC:
    ds.download_features(verbose=True)

    temp_rows = {}
    for item in ds.iter_items():
        if OBS_KEY not in item.get('observations_index', {}):
            continue
        key = (item['src'], item['loc_id'], item['year'])
        if key in temp_rows:
            continue
        ts = item[KEY_FEATURES]['temperature_2m_mean'].astype(float)
        temp_rows[key] = {
            'src': item['src'], 'loc_id': item['loc_id'], 'year': item['year'],
            'mean_winter_T': float(ts[:120].mean()),
            'mean_spring_T': float(ts[120:212].mean()),
            'chill_days':    float(func_chilling_days(ts).sum()),
            'forcing_gdu':   float(np.maximum(ts - 4.0, 0).sum()),
        }

    df_temp = pd.DataFrame(list(temp_rows.values()))
    df = df.merge(df_temp, on=['src', 'loc_id', 'year'], how='left')
    HAS_TEMP = True
    print(f'Temperature joined for {df["mean_winter_T"].notna().sum()} / {len(df)} rows')

elif not USE_SYNTHETIC:
    print('DOWNLOAD_TEMPERATURE = False — skipping temperature sections.')


## 2. Spatial coverage

Map of all stations coloured by region. A CF model requires stations across
the full climatic gradient of Japan — from cold Hokkaido to subtropical Okinawa.

In [ ]:
loc_df = df.groupby('loc_id').agg(
    lat=('lat','first'), lon=('lon','first'), alt=('alt','first'),
    region=('region','first'), loc_short=('loc_short','first'),
    n=('bloom_ix','count'),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.suptitle('Station coverage — GMU Cherry Japan (Yedoensis)',
             fontsize=12, fontweight='bold')

# — map ────────────────────────────────────────────────────────────────
ax = axes[0]
for region in REGION_ORDER:
    sub = loc_df[loc_df['region'] == region]
    if sub.empty: continue
    ax.scatter(sub['lon'], sub['lat'],
               c=[REGION_PALETTE[region]], s=sub['n'] * 1.5 + 20,
               alpha=0.85, label=f'{region} ({len(sub)})',
               edgecolors='white', linewidths=0.5)
ax.set_xlabel('Longitude', fontsize=9)
ax.set_ylabel('Latitude', fontsize=9)
ax.set_title('Station map  (size ∝ n years)', fontsize=9, fontweight='bold')
ax.legend(fontsize=7, framealpha=0.9, ncol=1)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# — latitude distribution by region ────────────────────────────────────
ax = axes[1]
lat_bins_plot = np.linspace(df['lat'].min() - 1, df['lat'].max() + 1, 25)
for region in REGION_ORDER:
    sub = df[df['region'] == region]['lat']
    if sub.empty: continue
    ax.hist(sub, bins=lat_bins_plot, alpha=0.55,
            color=REGION_PALETTE[region], label=region, edgecolor='none')
ax.set_xlabel('Latitude (°N)', fontsize=9)
ax.set_ylabel('Sample count', fontsize=9)
ax.set_title('Latitude distribution by region', fontsize=9, fontweight='bold')
ax.legend(fontsize=7, framealpha=0.9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# — altitude distribution ───────────────────────────────────────────────
ax = axes[2]
ax.scatter(loc_df['lat'], loc_df['alt'],
           c=[REGION_PALETTE[r] for r in loc_df['region']],
           s=60, alpha=0.8, edgecolors='white', linewidths=0.5)
ax.set_xlabel('Latitude (°N)', fontsize=9)
ax.set_ylabel('Altitude (m)', fontsize=9)
ax.set_title('Latitude vs altitude', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print('Stations per region:')
print(loc_df.groupby('region')['loc_id'].count()
           .reindex([r for r in REGION_ORDER if r in loc_df['region'].values]))


## 3. Temporal completeness

A location × year presence matrix. Gaps (white cells) reduce the power to
estimate year effects; stations that drop out after a certain year cannot
contribute to trend estimation.

In [ ]:
mat_pres = df.pivot_table(
    index='loc_short', columns='year', values='bloom_ix',
    aggfunc='count', fill_value=0
).clip(upper=1)

# Order rows by latitude (north → south)
lat_order = (
    df.groupby('loc_short')['lat'].first()
      .sort_values(ascending=False)
      .index.tolist()
)
mat_pres = mat_pres.reindex(lat_order)

fig, ax = plt.subplots(figsize=(16, 9))
ax.imshow(mat_pres.values, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(mat_pres.shape[1]))
ax.set_xticklabels(mat_pres.columns, rotation=90, fontsize=7)
ax.set_yticks(range(mat_pres.shape[0]))
ax.set_yticklabels(mat_pres.index, fontsize=7)
ax.set_xlabel('Year', fontsize=9)
ax.set_title('Location × year presence  (blue = observed, white = missing)\n'
             'Rows ordered north → south by latitude',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

completeness = mat_pres.values.mean()
print(f'Overall completeness: {completeness:.1%}')
print(f'Locations with all years: '
      f'{(mat_pres.sum(axis=1) == mat_pres.shape[1]).sum()}')
print(f'Years with all locations: '
      f'{(mat_pres.sum(axis=0) == mat_pres.shape[0]).sum()}')


## 4. Bloom timing distribution

Distribution of bloom day-of-season (days from Oct 1) by region.
Wide, overlapping distributions across regions are needed to constrain
a CF model — if each region only contributes a narrow slice of the
bloom-timing range, the model cannot separate regional from climatic effects.

In [ ]:
regions_present = [r for r in REGION_ORDER if r in df['region'].values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Bloom timing (days from Oct 1) by region',
             fontsize=12, fontweight='bold')

# — violin by region ────────────────────────────────────────────────────
ax = axes[0]
data = [df[df['region'] == r]['bloom_ix'].values for r in regions_present]
parts = ax.violinplot(data, positions=range(len(regions_present)),
                      showmedians=True, showextrema=False)
for pc, r in zip(parts['bodies'], regions_present):
    pc.set_facecolor(REGION_PALETTE[r]); pc.set_alpha(0.7)
parts['cmedians'].set_color('black')
ax.set_xticks(range(len(regions_present)))
ax.set_xticklabels(regions_present, rotation=35, ha='right', fontsize=8.5)
ax.set_ylabel('Bloom day from Oct 1', fontsize=9)
ax.set_title('Bloom timing violin by region', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# — bloom timing vs latitude scatter ───────────────────────────────────
ax = axes[1]
for r in regions_present:
    sub = df[df['region'] == r].sample(
        min(200, len(df[df['region']==r])), random_state=0)
    ax.scatter(sub['lat'], sub['bloom_ix'],
               c=[REGION_PALETTE[r]], s=8, alpha=0.45, label=r)
coef = np.polyfit(df['lat'], df['bloom_ix'], 1)
x_line = np.linspace(df['lat'].min(), df['lat'].max(), 100)
ax.plot(x_line, np.polyval(coef, x_line), 'k--', lw=1.5,
        label=f'Slope: {coef[0]:.1f} days/°N')
r_lat = df[['lat','bloom_ix']].corr().iloc[0, 1]
ax.set_xlabel('Latitude (°N)', fontsize=9)
ax.set_ylabel('Bloom day from Oct 1', fontsize=9)
ax.set_title(f'Bloom vs latitude  (r = {r_lat:.3f})',
             fontsize=9, fontweight='bold')
ax.legend(fontsize=7, ncol=2, framealpha=0.9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


## 5. Climate range per region

For a CF model to be identifiable, the dataset must span a wide range of
winter chilling and spring forcing. If all stations experience similar
chilling, the threshold parameter `threshold_c` cannot be estimated.

In [ ]:
if not HAS_TEMP:
    print('Set DOWNLOAD_TEMPERATURE = True and re-run.')
else:
    df['climate_bin'] = pd.cut(df['mean_winter_T'],
                                bins=CLIMATE_BINS, labels=CLIMATE_LABELS)
    df['spring_bin']  = pd.cut(df['mean_spring_T'],
                                bins=SPRING_BINS,  labels=SPRING_LABELS)

    # ── Helper ───────────────────────────────────────────────────────────
    def _heatmap(ax, mat, title, cmap='YlOrRd', rank_label=True,
                 row_order=None, fontsize_cell=7):
        if row_order is not None:
            mat = mat.reindex(row_order)
        vals = mat.fillna(0).values.astype(float)
        im = ax.imshow(vals, cmap=cmap, aspect='auto')
        plt.colorbar(im, ax=ax, shrink=0.75, label='Sample count')
        ax.set_xticks(range(mat.shape[1]))
        ax.set_yticks(range(mat.shape[0]))
        ax.set_xticklabels(mat.columns, rotation=45, ha='right', fontsize=7.5)
        ax.set_yticklabels(mat.index, fontsize=8)
        for i in range(vals.shape[0]):
            for j in range(vals.shape[1]):
                v = int(vals[i, j])
                if v > 0:
                    ax.text(j, i, str(v), ha='center', va='center',
                            fontsize=fontsize_cell, color='black')
        if rank_label and vals.max() > 0:
            _, s, _ = np.linalg.svd(vals / (vals.sum() + 1e-9))
            explained = s[0]**2 / (s**2).sum()
            ax.set_title(f'{title}\n(top SV explains {explained:.0%} of variance)',
                         fontsize=9, fontweight='bold')
        else:
            ax.set_title(title, fontsize=9, fontweight='bold')

    # ── Region × climate bin heatmaps ────────────────────────────────────
    mat_winter = df.pivot_table(index='region', columns='climate_bin',
                                 values='bloom_ix', aggfunc='count', fill_value=0)
    mat_spring = df.pivot_table(index='region', columns='spring_bin',
                                 values='bloom_ix', aggfunc='count', fill_value=0)

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle('Region × climate bin coverage  (rank-1 → confounded)',
                 fontsize=12, fontweight='bold')
    _heatmap(axes[0], mat_winter, 'region × winter T bin (Oct–Jan)',
             row_order=[r for r in REGION_ORDER if r in mat_winter.index])
    axes[0].set_xlabel('Mean winter temperature bin (°C)', fontsize=9)
    axes[0].set_ylabel('Region (north → south)', fontsize=9)
    _heatmap(axes[1], mat_spring, 'region × spring T bin (Feb–Apr)',
             row_order=[r for r in REGION_ORDER if r in mat_spring.index])
    axes[1].set_xlabel('Mean spring temperature bin (°C)', fontsize=9)
    axes[1].set_ylabel('', fontsize=9)
    plt.tight_layout()
    plt.show()

    # ── 2D climate niche scatter with 90% ellipses ───────────────────────
    from matplotlib.patches import Ellipse

    fig, ax = plt.subplots(figsize=(9, 6.5))
    fig.suptitle('2D climate niche — winter T × spring T by region',
                 fontsize=12, fontweight='bold')
    for r in regions_present:
        sub = df[df['region'] == r].dropna(
            subset=['mean_winter_T', 'mean_spring_T'])
        ax.scatter(sub['mean_winter_T'], sub['mean_spring_T'],
                   c=[REGION_PALETTE[r]], s=10, alpha=0.4, label=r)
        if len(sub) < 10:
            continue
        x, y = sub['mean_winter_T'].values, sub['mean_spring_T'].values
        cov  = np.cov(x, y)
        ev, vecs = np.linalg.eigh(cov)
        ev, vecs = ev[ev.argsort()[::-1]], vecs[:, ev.argsort()[::-1]]
        angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
        w90, h90 = 2 * np.sqrt(4.605 * ev)   # chi2 2-dof 90%
        ell = Ellipse((x.mean(), y.mean()), w90, h90, angle=angle,
                      edgecolor=REGION_PALETTE[r], facecolor='none',
                      lw=1.5, ls='--', alpha=0.85)
        ax.add_patch(ell)
    ax.set_xlabel('Mean winter temperature (Oct–Jan, °C)', fontsize=10)
    ax.set_ylabel('Mean spring temperature (Feb–Apr, °C)', fontsize=10)
    ax.legend(fontsize=8, ncol=2, framealpha=0.9,
              title='Region  (dashed ellipse = 90% of obs)')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

    # ── Boxplots ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('CF-model driver distributions by region',
                 fontsize=12, fontweight='bold')
    plot_vars = [
        ('mean_winter_T', 'Mean winter temperature (°C)', 'Oct–Jan mean T'),
        ('chill_days',    'Total chill days (T ≤ 7.2 °C)', 'Season chill days'),
        ('forcing_gdu',   'Total forcing GDU (T > 4 °C)',  'Season forcing GDU'),
    ]
    for ax, (col, xlabel, title) in zip(axes, plot_vars):
        data = [df[df['region'] == r][col].dropna().values
                for r in regions_present]
        bp = ax.boxplot(data, patch_artist=True,
                        medianprops=dict(color='black', lw=1.8),
                        whiskerprops=dict(lw=1), capprops=dict(lw=1),
                        flierprops=dict(marker='o', markersize=2,
                                        alpha=0.3, markeredgewidth=0))
        for patch, r in zip(bp['boxes'], regions_present):
            patch.set_facecolor(REGION_PALETTE[r]); patch.set_alpha(0.75)
        all_vals = df[col].dropna()
        ax.axhspan(all_vals.quantile(0.25), all_vals.quantile(0.75),
                   color='lightgrey', alpha=0.25, zorder=0,
                   label='All-regions IQR')
        ax.set_xticks(range(1, len(regions_present) + 1))
        ax.set_xticklabels(regions_present, rotation=35, ha='right', fontsize=8)
        ax.set_ylabel(xlabel, fontsize=9)
        ax.set_title(title, fontsize=9, fontweight='bold')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## 6. Driver range — identifiability of CF model parameters

Scatter of chilling vs forcing and both vs bloom timing. Wide, continuous
coverage of the driver space is needed to separately estimate `threshold_c`
and `threshold_f`.

In [ ]:
if not HAS_TEMP:
    print('Set DOWNLOAD_TEMPERATURE = True and re-run.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    fig.suptitle('Driver range analysis — identifiability of CF model parameters',
                 fontsize=12, fontweight='bold')

    for ax, (xcol, ycol, xlabel, ylabel) in zip(axes, [
            ('chill_days', 'forcing_gdu',
             'Total chill days (T ≤ 7.2 °C)', 'Total forcing GDU (T > 4 °C)'),
            ('chill_days', 'bloom_ix',
             'Total chill days', 'Bloom day from Oct 1'),
            ('forcing_gdu', 'bloom_ix',
             'Total forcing GDU', 'Bloom day from Oct 1'),
    ]):
        for r in regions_present:
            sub = df[df['region'] == r].dropna(subset=[xcol, ycol])
            ax.scatter(sub[xcol], sub[ycol],
                       c=[REGION_PALETTE[r]], s=10, alpha=0.4, label=r)
        corr = df[[xcol, ycol]].dropna().corr().iloc[0, 1]
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(f'r = {corr:.3f}', fontsize=9, fontweight='bold')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    axes[0].legend(fontsize=7, ncol=2, framealpha=0.9)
    plt.tight_layout()
    plt.show()

    print('Driver ranges:')
    for col, label in [('chill_days','Chill days'),('forcing_gdu','Forcing GDU')]:
        vals = df[col].dropna()
        print(f'  {label}: {vals.min():.0f} – {vals.max():.0f}'
              f'  (mean {vals.mean():.0f}, std {vals.std():.0f})')


## 7. Temporal variation

Year-to-year variation provides the signal for estimating year effects and
validating the model across different climatic conditions.

In [ ]:
yr_clim = df.groupby('year').agg(
    bloom_mean=('bloom_ix', 'mean'),
    bloom_std=('bloom_ix', 'std'),
    n=('bloom_ix', 'count'),
).reset_index()
yr_clim['bloom_anom'] = yr_clim['bloom_mean'] - yr_clim['bloom_mean'].mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Temporal variation', fontsize=12, fontweight='bold')

# — bloom trend by region ─────────────────────────────────────────────
ax = axes[0]
for r in regions_present:
    sub = df[df['region'] == r].groupby('year')['bloom_ix'].median()
    rolled = sub.rolling(5, center=True, min_periods=2).median()
    ax.plot(rolled.index, rolled.values,
            color=REGION_PALETTE[r], lw=1.5, label=r)
ax.set_xlabel('Year', fontsize=9)
ax.set_ylabel('Median bloom day (5-yr rolling)', fontsize=9)
ax.set_title('Bloom trends by region', fontsize=9, fontweight='bold')
ax.legend(fontsize=7, ncol=2, framealpha=0.9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# — annual bloom anomaly ──────────────────────────────────────────────
ax = axes[1]
bar_colors = ['#e74c3c' if v > 0 else '#2980b9' for v in yr_clim['bloom_anom']]
ax.bar(yr_clim['year'], yr_clim['bloom_anom'], color=bar_colors, alpha=0.85)
ax.axhline(0, color='grey', lw=0.8)
ax.set_xlabel('Year', fontsize=9)
ax.set_ylabel('Bloom anomaly (days)', fontsize=9)
ax.set_title('Cross-station bloom anomaly\n(red = late, blue = early)',
             fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# — interannual variability per station ───────────────────────────────
ax = axes[2]
iav = df.groupby('loc_short')['bloom_ix'].std().sort_values()
ax.barh(range(len(iav)), iav.values, color='steelblue', alpha=0.8)
ax.set_yticks(range(len(iav)))
ax.set_yticklabels(iav.index, fontsize=6)
ax.axvline(iav.median(), color='crimson', lw=1.5, ls='--',
           label=f'Median = {iav.median():.1f} days')
ax.set_xlabel('Interannual std of bloom day (days)', fontsize=9)
ax.set_title('Interannual variability per station',
             fontsize=9, fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

q33, q67 = yr_clim['bloom_anom'].quantile([0.33, 0.67])
early = yr_clim[yr_clim['bloom_anom'] < q33]['year'].tolist()
late  = yr_clim[yr_clim['bloom_anom'] > q67]['year'].tolist()
print(f'Early years (bottom third): {sorted(early)}')
print(f'Late  years (top third):    {sorted(late)}')
print(f'Bloom range: {df["bloom_ix"].min()} – {df["bloom_ix"].max()} days '
      f'(spread {df["bloom_ix"].max() - df["bloom_ix"].min()} days)')


## 8. Summary: adequacy verdict

In [ ]:
n_locs    = df['loc_id'].nunique()
n_years   = df['year'].nunique()
yr_range  = df['year'].max() - df['year'].min()
complete  = mat_pres.values.mean()
bloom_std = df['bloom_ix'].std()
n_regions = df['region'].nunique()

rows = [
    ('Total observations',       f'{len(df):,}',
                                  'Should be >> n_params'),
    ('Locations',                f'{n_locs}',              '—'),
    ('Years',                    f'{n_years}  ({yr_range} yr span)',
                                  'Need interannual variation'),
    ('Regions covered',          f'{n_regions}',           '—'),
    ('Network completeness',     f'{complete:.1%}',
                                  '> 80% recommended'),
    ('Bloom day range (all)',    f'{df["bloom_ix"].min()} – {df["bloom_ix"].max()} days',
                                  'Larger spread = more phenological signal'),
    ('Bloom std (all)',          f'{bloom_std:.1f} days',
                                  'Larger = more signal'),
    ('Lat–bloom correlation',    f'{df[["lat","bloom_ix"]].corr().iloc[0,1]:.3f}',
                                  'High r → latitude explains much of the variation'),
]

if HAS_TEMP:
    chill_range = df['chill_days'].max() - df['chill_days'].min()
    fgdu_range  = df['forcing_gdu'].max() - df['forcing_gdu'].min()
    wT_range    = df['mean_winter_T'].max() - df['mean_winter_T'].min()
    rows += [
        ('Winter T range',       f'{df["mean_winter_T"].min():.1f} – '
                                  f'{df["mean_winter_T"].max():.1f} °C  '
                                  f'(spread {wT_range:.1f} °C)',
                                  '> 8 °C recommended for CF models'),
        ('Chill days range',     f'{df["chill_days"].min():.0f} – '
                                  f'{df["chill_days"].max():.0f}  '
                                  f'(spread {chill_range:.0f})',
                                  'Wide range constrains threshold_c'),
        ('Forcing GDU range',    f'{df["forcing_gdu"].min():.0f} – '
                                  f'{df["forcing_gdu"].max():.0f}  '
                                  f'(spread {fgdu_range:.0f})',
                                  'Wide range constrains threshold_f'),
    ]

display(pd.DataFrame(rows, columns=['Diagnostic', 'Value', 'Guideline']))
